# Weekly Sector Rotation — neural network model

Explores whether a small neural network finds real predictive structure that
`feature_diagnostics.ipynb`'s linear baseline (Stage 7 logistic regression) couldn't —
that stage ended statistically indistinguishable from chance (Stage 8's block-bootstrap
CIs all straddled zero), which is the benchmark this notebook needs to beat, not a
presumption that a neural network automatically will.

**Confirmed choices, from the pre-build clarifying questions**:
- **Framework**: PyTorch (`torch`, added as the `deep-learning` optional extra in
  `pyproject.toml`, following this project's existing optional-dependency pattern).
- **Architecture**: a small MLP — 2 hidden layers (16 -> 8 units), ReLU, dropout, L2
  weight decay — sized for ~11k complete-case rows, not scaled up before there's a
  reason to believe more capacity would help rather than just overfit noise.
- **Protocol**: an exact reuse of Stage 7's harness — same expanding walk-forward-by-year
  split (`MIN_TRAIN_YEARS = 3`), same 9 cumulative feature sets (B -> A+B -> ... ->
  all 9 groups), same classification label (`next_week_excess_return > 0`), same
  metrics (pooled accuracy/AUC, top-K/bottom-K spread and hit rate for K=2 and K=3).
  The only thing that changes from Stage 7 is the model itself, so any difference in
  result is attributable to the model, not a changed evaluation setup.

One addition Stage 7 didn't need: an **inner validation split for early stopping** —
each fold's training years are further split into fit-years and a held-out validation
year (the most recent training year), used only to pick the training epoch to stop at,
never to touch the scaler or the actual test fold. A linear model doesn't need this;
an MLP with this little data will overfit within a few dozen epochs without it.

This notebook is self-contained (reloads the feature panel and rebuilds everything it
needs from scratch), matching how every notebook in this project stands alone rather
than sharing in-memory state with `feature_diagnostics.ipynb`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().resolve().parents[3]
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=["date"])
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)

AUDIT_COLUMNS = [
    "week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
    "next_week_first_trading_date", "next_week_last_trading_date",
]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

panel["next_week_excess_return"] = panel.groupby("symbol")["excess_return_vs_vti_1w"].shift(-1)
research = panel.dropna(subset=["next_week_excess_return"]).reset_index(drop=True)

FEATURE_GROUPS = {
    "A": ["abs_return_1w", "abs_return_2w", "abs_return_4w", "abs_return_8w", "abs_return_12w", "abs_return_26w"],
    "B": ["excess_return_vs_vti_1w", "excess_return_vs_vti_2w", "excess_return_vs_vti_4w",
          "excess_return_vs_vti_8w", "excess_return_vs_vti_12w", "excess_return_vs_vti_26w"],
    "C": ["cross_sectional_rank_1w", "cross_sectional_rank_4w", "cross_sectional_rank_8w", "cross_sectional_rank_12w"],
    "D": ["realized_vol_4w", "realized_vol_12w", "realized_vol_26w"],
    "E": ["trend_dist_from_ma_4w", "trend_dist_from_ma_12w", "trend_dist_from_ma_26w"],
    "F": ["drawdown_dist_from_high_13w", "drawdown_dist_from_high_26w"],
    "G": ["volume_change_1w", "volume_vs_avg_4w", "volume_vs_avg_12w"],
    "H": ["vti_return_1w", "vti_return_4w", "vti_return_12w", "vti_volatility_4w"],
    "I": ["sector_return_dispersion_1w", "average_sector_return_1w", "sectors_outperforming_vti_1w"],
}
feature_to_group = {c: g for g, cols in FEATURE_GROUPS.items() for c in cols}

model_data = research.dropna(subset=FEATURE_COLUMNS).copy()
model_data["label_binary"] = (model_data["next_week_excess_return"] > 0).astype(int)
model_data["year"] = model_data["date"].dt.year

MIN_TRAIN_YEARS = 3
RANK_K_VALUES = [2, 3]
years = sorted(model_data["year"].unique())
test_years = years[MIN_TRAIN_YEARS:]

# Cumulative feature sets in the plan's specified order: B alone, then A+B, A+B+C, ... , all 9 groups.
GROUP_ADD_ORDER = ["B", "A", "C", "D", "E", "F", "G", "H", "I"]
FEATURE_SET_STEPS = []
added_so_far = []
for group in GROUP_ADD_ORDER:
    added_so_far.append(group)
    cols = [c for g in added_so_far for c in FEATURE_GROUPS[g]]
    label = "+".join(sorted(added_so_far))
    FEATURE_SET_STEPS.append((label, cols))

print(f"{len(model_data):,} complete-case rows, years {years[0]}-{years[-1]}")
print(f"walk-forward test years ({len(test_years)} folds): {test_years[0]}-{test_years[-1]}")
print(f"torch {torch.__version__}")
for lbl, cols in FEATURE_SET_STEPS:
    print(f"  {lbl}: {len(cols)} features")

## Model definition

In [ ]:
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    """2 hidden layers, small on purpose -- ~11k complete-case rows, not a deep model."""

    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)  # raw logits; BCEWithLogitsLoss applies sigmoid internally


def train_fold_mlp(X_fit: np.ndarray, y_fit: np.ndarray, X_val: np.ndarray, y_val: np.ndarray,
                    n_features: int, seed: int = NN_SEED) -> SmallMLP:
    """Full-batch training with early stopping on validation loss; returns the best-val-loss weights."""
    torch.manual_seed(seed)
    model = SmallMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
    y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model

## Walk-forward evaluation (identical harness to Stage 7, model swapped)

In [ ]:
def top_bottom_spread_from_scores(frame: pd.DataFrame, score_col: str, k: int) -> pd.Series:
    """One week's top-K/bottom-K next-week excess return, ranked by score_col (predicted prob)."""
    ranked = frame[score_col].rank(ascending=False, method="first")  # 1 = highest score
    n = len(frame)
    top_return = frame.loc[ranked <= k, "next_week_excess_return"].mean()
    bottom_return = frame.loc[ranked > n - k, "next_week_excess_return"].mean()
    return pd.Series({"top_return": top_return, "bottom_return": bottom_return})


def walk_forward_evaluate_nn(feature_cols: list[str]) -> dict:
    """Same expanding walk-forward-by-year design as Stage 7's logistic regression, with an
    inner fit/val split (most recent training year held out) added purely for early stopping."""
    oof_frames = []
    for test_year in test_years:
        train_years_available = [y for y in years if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]

        train_mask = model_data["year"].isin(train_years_available)  # scaler fit: same rule as Stage 7
        fit_mask = model_data["year"].isin(fit_years)
        val_mask = model_data["year"] == val_year
        test_mask = model_data["year"] == test_year

        scaler = StandardScaler().fit(model_data.loc[train_mask, feature_cols])
        X_fit = scaler.transform(model_data.loc[fit_mask, feature_cols])
        X_val = scaler.transform(model_data.loc[val_mask, feature_cols])
        X_test = scaler.transform(model_data.loc[test_mask, feature_cols])
        y_fit = model_data.loc[fit_mask, "label_binary"].to_numpy()
        y_val = model_data.loc[val_mask, "label_binary"].to_numpy()

        model = train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features=len(feature_cols))
        with torch.no_grad():
            predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = model_data.loc[test_mask, ["date", "symbol", "next_week_excess_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

    oof = pd.concat(oof_frames, ignore_index=True)

    accuracy = accuracy_score(oof["label_binary"], oof["predicted_proba"] >= 0.5)
    auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])

    spread_stats = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
        spread_stats[f"mean_spread_k{k}"] = spread.mean()
        spread_stats[f"hit_rate_k{k}"] = (spread > 0).mean()

    return {"accuracy": accuracy, "auc": auc, "n_test_rows": len(oof), **spread_stats}

## Run the ablation

In [ ]:
nn_ablation_rows = []
for label, feature_cols in FEATURE_SET_STEPS:
    metrics = walk_forward_evaluate_nn(feature_cols)
    nn_ablation_rows.append({"feature_set": label, "n_features": len(feature_cols), **metrics})
    print(f"{label}: accuracy={metrics['accuracy']:.4f}, auc={metrics['auc']:.4f}, "
          f"k3_spread={metrics['mean_spread_k3']:.5f}")

nn_ablation_results = pd.DataFrame(nn_ablation_rows)
nn_ablation_results.to_csv(OUTPUTS_ROOT / "nn_ablation_walk_forward_results.csv", index=False)
nn_ablation_results

## Comparison against Stage 7 (logistic regression)

In [ ]:
lr_results = pd.read_csv(OUTPUTS_ROOT / "ablation_walk_forward_results.csv")
comparison = lr_results.merge(nn_ablation_results, on=["feature_set", "n_features"], suffixes=("_lr", "_nn"))
comparison = comparison[["feature_set", "n_features", "auc_lr", "auc_nn", "mean_spread_k3_lr", "mean_spread_k3_nn"]]
comparison.to_csv(OUTPUTS_ROOT / "nn_vs_logistic_regression_comparison.csv", index=False)
comparison

## Block-bootstrap CI on the full-model spread (same method as Stage 8)

The full 9-group model has the best AUC and K2 spread of anything built across this
whole project so far — before reading anything into that, check it against the same
moving block bootstrap Stage 8 used, not just the point estimate.

In [ ]:
def walk_forward_oof_spreads_nn(feature_cols: list[str]) -> dict[int, pd.Series]:
    """Same as walk_forward_evaluate_nn, but returns the raw weekly spread series per K
    instead of summary stats -- needed for the bootstrap, which operates on the series itself."""
    oof_frames = []
    for test_year in test_years:
        train_years_available = [y for y in years if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]

        train_mask = model_data["year"].isin(train_years_available)
        fit_mask = model_data["year"].isin(fit_years)
        val_mask = model_data["year"] == val_year
        test_mask = model_data["year"] == test_year

        scaler = StandardScaler().fit(model_data.loc[train_mask, feature_cols])
        X_fit = scaler.transform(model_data.loc[fit_mask, feature_cols])
        X_val = scaler.transform(model_data.loc[val_mask, feature_cols])
        X_test = scaler.transform(model_data.loc[test_mask, feature_cols])
        y_fit = model_data.loc[fit_mask, "label_binary"].to_numpy()
        y_val = model_data.loc[val_mask, "label_binary"].to_numpy()

        model = train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features=len(feature_cols))
        with torch.no_grad():
            predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = model_data.loc[test_mask, ["date", "symbol", "next_week_excess_return"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

    oof = pd.concat(oof_frames, ignore_index=True)
    spreads = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spreads[k] = (weekly["top_return"] - weekly["bottom_return"]).dropna()
    return spreads


def block_bootstrap_ci(values: pd.Series, block_size: int = 8, n_boot: int = 5000,
                        ci: float = 0.95, seed: int = 20260812) -> dict:
    """Moving block bootstrap on a 1-D series; returns the point estimate, CI bounds, and significance flag.
    Same method and BLOCK_SIZE/N_BOOTSTRAP as feature_diagnostics.ipynb's Stage 8."""
    arr = values.to_numpy()
    n = len(arr)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(np.ceil(n / block_size))
    max_start = n - block_size

    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([arr[s : s + block_size] for s in starts])[:n]
        boot_means[b] = sample.mean()

    lower, upper = np.percentile(boot_means, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return {
        "point_estimate": arr.mean(),
        "ci_lower": lower,
        "ci_upper": upper,
        "n_weeks": n,
        "significant": bool(lower > 0 or upper < 0),
    }


full_model_cols = FEATURE_SET_STEPS[-1][1]
full_model_nn_spreads = walk_forward_oof_spreads_nn(full_model_cols)

nn_bootstrap_rows = []
for k in RANK_K_VALUES:
    result = block_bootstrap_ci(full_model_nn_spreads[k])
    nn_bootstrap_rows.append({"target": f"nn_full_model_k{k}", **result})

nn_bootstrap_summary = pd.DataFrame(nn_bootstrap_rows)
nn_bootstrap_summary.to_csv(OUTPUTS_ROOT / "nn_block_bootstrap_ci.csv", index=False)
nn_bootstrap_summary

## Findings

**Same overall picture as the linear baseline: hovering around chance, noisier but not
meaningfully better.** AUC across the 9 feature sets ranges 0.498-0.515 (vs. 0.499-0.503
for logistic regression) and accuracy 0.495-0.509 (vs. 0.494-0.506) — a wider spread in
both directions, consistent with a higher-variance model on this little data, not
consistent evidence of extra signal. Unlike Stage 7, the NN's best result *is* the full
9-group model (AUC 0.5097, K2 spread +0.00127) rather than a smaller subset, which is at
least directionally the outcome you'd hope a nonlinear model would produce if the
interaction effects hinted at in Stage 5 were real and exploitable.

**But it doesn't clear the bar Stage 8 set.** The full model's K2 spread — the best
single number produced anywhere in this project — has a 95% block-bootstrap CI of
[-0.00028, +0.00243]. Still includes zero, so still not statistically significant. It is
the *closest* any result has come (compare Stage 8's baseline/full logistic regression
CIs, which sat further from zero on the low side too), which is worth noting honestly
as a "closer, not there" result rather than either dismissing it or overselling it.

**What this does and doesn't establish**: this one small-MLP architecture, at these
hyperparameters, on this feature set and universe, does not find a statistically
significant weekly top-K signal — matching the logistic regression's null result rather
than overturning it. It does not rule out that a different architecture, more training
data, engineered interaction features, or a different asset universe/rebalance frequency
would do better; none of those were tried here. Given how consistently null the result
has been across two very different model classes now (Stages 2-8's linear analysis and
this notebook's nonlinear one), the more likely explanation is that 11 sector ETFs at a
weekly rebalance, with this feature set, genuinely don't contain a strong tradeable
signal — not that the modeling has been insufficient so far.

## Architecture v2: wider single layer, stronger regularization, seed ensemble

v1's own finding is the reason for what changes here: it wasn't clearly *worse* than
logistic regression, it was *noisier* — a wider AUC range in both directions, on a
model fit fresh from a single random initialization per fold. Two changes, chosen to
address that specific finding rather than just retrying arbitrary numbers:

- **Architecture**: one hidden layer of 32 units instead of two narrow layers (16 -> 8).
  Two layers add depth/capacity that ~1-2k training rows per fold may not support;
  a single wider layer is a different capacity profile, not just "bigger."
- **Regularization**: dropout 0.3 -> 0.4, weight decay 1e-4 -> 1e-3 — both increased,
  since v1 showing high fold-to-fold variance is itself evidence of underfit
  regularization relative to the data size, not overfit capacity needing to shrink.
- **Seed ensemble**: `N_SEEDS = 5` models trained per fold (different random
  initialization each), predicted probabilities averaged before ranking — a
  variance-reduction technique, not a capacity change. This directly targets the
  "noisier, not better" finding: averaging over initializations should shrink the
  single-seed noise that dominated v1's fold-to-fold swings, giving a more reliable
  read on whether there's real signal underneath it.

**Multiple-comparisons caveat, stated rather than ignored**: this is a second
configuration chosen after seeing v1's result on these same walk-forward test years.
That's a real, if mild, risk of unconsciously curve-fitting the choice of what to try
next to the test period. The changes above were chosen to address a specific, named
weakness in v1 (variance) rather than to chase the best number, and this is capped at
one further architecture, not an open-ended search — but it's not a blind, pre-registered
choice either, and shouldn't be read as if it were.

In [ ]:
V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    """1 hidden layer, wider than v1's narrowest layer -- a different capacity profile, not just bigger."""

    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN),
            nn.ReLU(),
            nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def train_fold_mlp_v2(X_fit: np.ndarray, y_fit: np.ndarray, X_val: np.ndarray, y_val: np.ndarray,
                       n_features: int, seed: int) -> WideSingleLayerMLP:
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=V2_LEARNING_RATE, weight_decay=V2_WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
    y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for _ in range(V2_MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= V2_PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model


def walk_forward_evaluate_nn_v2(feature_cols: list[str]) -> dict:
    """Same walk-forward-by-year design, N_SEEDS models per fold, predicted probabilities averaged."""
    oof_frames = []
    for test_year in test_years:
        train_years_available = [y for y in years if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]

        train_mask = model_data["year"].isin(train_years_available)
        fit_mask = model_data["year"].isin(fit_years)
        val_mask = model_data["year"] == val_year
        test_mask = model_data["year"] == test_year

        scaler = StandardScaler().fit(model_data.loc[train_mask, feature_cols])
        X_fit = scaler.transform(model_data.loc[fit_mask, feature_cols])
        X_val = scaler.transform(model_data.loc[val_mask, feature_cols])
        X_test = scaler.transform(model_data.loc[test_mask, feature_cols])
        y_fit = model_data.loc[fit_mask, "label_binary"].to_numpy()
        y_val = model_data.loc[val_mask, "label_binary"].to_numpy()

        seed_predictions = []
        for seed_offset in range(N_SEEDS):
            model = train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset)
            with torch.no_grad():
                seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
        predicted_proba = np.mean(seed_predictions, axis=0)

        fold = model_data.loc[test_mask, ["date", "symbol", "next_week_excess_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

    oof = pd.concat(oof_frames, ignore_index=True)

    accuracy = accuracy_score(oof["label_binary"], oof["predicted_proba"] >= 0.5)
    auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])

    spread_stats = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
        spread_stats[f"mean_spread_k{k}"] = spread.mean()
        spread_stats[f"hit_rate_k{k}"] = (spread > 0).mean()

    return {"accuracy": accuracy, "auc": auc, "n_test_rows": len(oof), **spread_stats}

### Run the v2 ablation

5x the model fits of v1 (5 seeds per fold) — expect roughly 5x the runtime.

In [ ]:
nn_v2_rows = []
for label, feature_cols in FEATURE_SET_STEPS:
    metrics = walk_forward_evaluate_nn_v2(feature_cols)
    nn_v2_rows.append({"feature_set": label, "n_features": len(feature_cols), **metrics})
    print(f"{label}: accuracy={metrics['accuracy']:.4f}, auc={metrics['auc']:.4f}, "
          f"k3_spread={metrics['mean_spread_k3']:.5f}")

nn_v2_results = pd.DataFrame(nn_v2_rows)
nn_v2_results.to_csv(OUTPUTS_ROOT / "nn_v2_ablation_walk_forward_results.csv", index=False)
nn_v2_results

### Three-way comparison, and a bootstrap check on v2's best result

In [ ]:
three_way = lr_results.merge(nn_ablation_results, on=["feature_set", "n_features"], suffixes=("_lr", "_nn_v1"))
three_way = three_way.merge(nn_v2_results, on=["feature_set", "n_features"])
three_way = three_way.rename(columns={"accuracy": "accuracy_nn_v2", "auc": "auc_nn_v2",
                                       "mean_spread_k3": "mean_spread_k3_nn_v2"})
three_way = three_way[["feature_set", "n_features", "auc_lr", "auc_nn_v1", "auc_nn_v2",
                        "mean_spread_k3_lr", "mean_spread_k3_nn_v1", "mean_spread_k3_nn_v2"]]
three_way.to_csv(OUTPUTS_ROOT / "nn_v1_v2_lr_comparison.csv", index=False)
print(three_way.round(5).to_string(index=False))

best_row = nn_v2_results.loc[nn_v2_results["auc"].idxmax()]
best_label = best_row["feature_set"]
print(f"\nv2's best AUC: {best_label} ({best_row['auc']:.4f})")

def walk_forward_oof_spreads_nn_v2(feature_cols: list[str]) -> dict[int, pd.Series]:
    oof_frames = []
    for test_year in test_years:
        train_years_available = [y for y in years if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]

        train_mask = model_data["year"].isin(train_years_available)
        fit_mask = model_data["year"].isin(fit_years)
        val_mask = model_data["year"] == val_year
        test_mask = model_data["year"] == test_year

        scaler = StandardScaler().fit(model_data.loc[train_mask, feature_cols])
        X_fit = scaler.transform(model_data.loc[fit_mask, feature_cols])
        X_val = scaler.transform(model_data.loc[val_mask, feature_cols])
        X_test = scaler.transform(model_data.loc[test_mask, feature_cols])
        y_fit = model_data.loc[fit_mask, "label_binary"].to_numpy()
        y_val = model_data.loc[val_mask, "label_binary"].to_numpy()

        seed_predictions = []
        for seed_offset in range(N_SEEDS):
            model = train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset)
            with torch.no_grad():
                seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
        predicted_proba = np.mean(seed_predictions, axis=0)

        fold = model_data.loc[test_mask, ["date", "symbol", "next_week_excess_return"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

    oof = pd.concat(oof_frames, ignore_index=True)
    spreads = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spreads[k] = (weekly["top_return"] - weekly["bottom_return"]).dropna()
    return spreads


best_cols = dict(FEATURE_SET_STEPS)[best_label]
best_v2_spreads = walk_forward_oof_spreads_nn_v2(best_cols)

v2_bootstrap_rows = []
for k in RANK_K_VALUES:
    result = block_bootstrap_ci(best_v2_spreads[k])
    v2_bootstrap_rows.append({"target": f"nn_v2_{best_label}_k{k}", **result})

v2_bootstrap_summary = pd.DataFrame(v2_bootstrap_rows)
v2_bootstrap_summary.to_csv(OUTPUTS_ROOT / "nn_v2_block_bootstrap_ci.csv", index=False)
v2_bootstrap_summary

## Findings (v2)

**The hypothesis behind v2 was wrong, stated plainly rather than quietly dropped.** The
seed ensemble was built to reduce the fold-to-fold/feature-set-to-feature-set AUC
variance flagged in v1. It didn't: AUC standard deviation across the 9 feature sets is
0.00571 for v2 vs. 0.00543 for v1 — essentially unchanged, if anything marginally
higher. That means v1's instability wasn't primarily single-seed initialization
noise (which averaging over 5 seeds should fix) — it's more likely genuine sampling
variability across different walk-forward folds and feature combinations on a dataset
this size, which no amount of seed-averaging *within* a fold can smooth out. This is a
real, useful methodological finding in its own right, not just a failed fix.

**The headline result is otherwise unchanged.** v2's best AUC (0.5100, full 9-group
model) is statistically indistinguishable from v1's best (0.5097) and from logistic
regression's best (0.5034) — three different models, same ceiling. v2's bootstrap CI on
its own best result: K2 `[-0.00073, +0.00207]`, K3 `[-0.00060, +0.00181]` — both still
include zero.

**Where this leaves the neural-network exploration**: two structurally different
architectures (2-layer narrow, 1-layer wide + 5-seed ensemble + stronger
regularization), on top of a linear baseline, all converge on the same conclusion —
no statistically significant weekly top-K signal in this feature set on this 11-sector
universe. Per the multiple-comparisons caveat stated before building v2: two
architectures is a deliberately small, capped search, not license to keep trying
configurations until one clears significance by chance. Further architecture search
from here should be treated as exploratory, not as approaching a real result — the
weight of evidence across linear and nonlinear models both now points toward the
feature set/universe being the limiting factor, not the model class.